# TP 09 | Un pipeline de classification sur le Titanic

Peut-on estimer la survie d’un passager à partir des informations connues à l’embarquement ? Une ligne représente un passager, la cible vaut 1 s’il a survécu et 0 sinon. C’est une **classification binaire**, sur un événement historique. Un bon score sur ce manifeste ne garantit rien pour un autre naufrage.

**Parcours.** 1. Charger les données / 2. Réserver le test / 3. Explorer / 4. Nettoyer / 5. Construire les variables / 6. Comparer les modèles / 7. Régler les hyperparamètres / 8. Évaluer / 9. Sauvegarder et rejouer.

Ouvrez le notebook dans VS Code avec le noyau Python du dépôt à jour (`uv sync --locked --extra spark`). Les données sont incluses. Exécutez les cellules dans l’ordre. Vous obtiendrez une comparaison de modèles, un bilan sur des billets réservés et un fichier capable de prédire à partir d’une nouvelle ligne brute.

Dans chaque cellule « À vous », remplacez les expressions `...`, puis relancez la cellule. Le contrôle indique quelle partie compléter. Les figures et les étapes suivantes utilisent vos résultats. Gardez le test fermé jusqu’à la section 8.

## 1. Charger le manifeste et définir les entrées

Le CSV local reprend les 891 lignes étiquetées du [jeu Titanic](https://www.kaggle.com/competitions/titanic/data). Les noms et cabines ont été retirés et chaque numéro de billet remplacé par un identifiant de groupe. On garde ce dernier pour construire les échantillons.

| Champ | Lecture |
| --- | --- |
| Pclass, Sex, Age | Classe du billet (1 à 3), sexe enregistré, âge en années |
| SibSp, Parch | Frères, sœurs ou conjoints à bord, puis parents ou enfants à bord |
| Fare, Embarked | Tarif enregistré et port (C = Cherbourg, Q = Queenstown, S = Southampton) |
| PassengerId, TicketGroup | Identifiant d’une personne et groupe partageant le billet |
| Survived | Verdict historique, réservé à l’apprentissage et à l’évaluation |

In [ ]:
# Lire le CSV fourni et vérifier qu'une ligne représente bien un passager.
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

root = Path.cwd()
if root.name == "corrections":
    root = root.parent
data = pd.read_csv(root / "donnees" / "titanic.csv")
print(f"{len(data)} lignes et {data.shape[1]} colonnes dans le manifeste.")

In [ ]:
# Examiner les entrées disponibles avant de séparer les billets.
inputs = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
assert data.PassengerId.is_unique and set(data.Survived.unique()) == {0, 1}
print(f"{len(data)} passagers, {data.TicketGroup.nunique()} billets distincts.")
display(data[inputs + ["TicketGroup"]].head())

## 2. Réserver le test avant d’explorer les résultats

Deux passagers d’un même billet partagent parfois famille, tarif et conditions de voyage. Si on les répartit au hasard ligne par ligne, la mesure peut être trop favorable. Notre unité de séparation est donc **le billet**. La stratification cherche à garder des proportions de survivants voisines sous cette contrainte.

On isole environ un cinquième du manifeste pour le test final. Dans le reste (le *train*), on prépare quatre plis : dans chaque pli, les billets de validation sont distincts des billets servant à ajuster le modèle. Chaque passager du train sert une fois à la validation. Le test reste à part pendant le choix du modèle.

![Répartition des passagers par usage et plis de validation](figures/recap-titanic-plis.svg)

En vert foncé, les passagers à prédire pour ce pli. En vert clair, les passagers qui servent à ajuster le pipeline. Le nombre sur chaque bande foncée donne son effectif.

In [ ]:
# Fixer un découpage reproductible et contrôler les frontières entre billets.
from sklearn.model_selection import StratifiedGroupKFold

seed = 42


def grouped_folds(frame, n_splits):
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(splitter.split(frame[inputs], frame.Survived, frame.TicketGroup))


dev_idx, test_idx = grouped_folds(data, 5)[0]
train = data.iloc[dev_idx].reset_index(drop=True)
test = data.iloc[test_idx].reset_index(drop=True)
assert set(train.TicketGroup).isdisjoint(test.TicketGroup)
print(f"Développement : {len(train)} personnes. Test réservé : {len(test)}.")

Un billet partagé peut-il se retrouver des deux côtés d’un pli ? Lisez la dernière colonne avant de comparer les tailles des quatre validations.

In [ ]:
# Compter les passagers et les billets communs de chaque paire train-validation.
folds = grouped_folds(train, 4)
X, y = train[inputs], train.Survived
bilan_plis = []
for numero, (a, b) in enumerate(folds, start=1):
    communs = set(train.iloc[a].TicketGroup) & set(train.iloc[b].TicketGroup)
    assert not communs
    bilan_plis.append((numero, len(a), len(b), len(communs)))
display(
    pd.DataFrame(
        bilan_plis, columns=["Pli", "Entraînement", "Validation", "Billets communs"]
    ).set_index("Pli")
)

## 3. Explorer les données d’entraînement

Quels champs sont incomplets ? La répartition des âges et la fréquence de survie sont-elles identiques dans toutes les classes de billet ? Regardez à la fois les **effectifs** et les **proportions**. Un taux décrit une association dans ce manifeste, sans isoler un effet causal.

![Âges disponibles ou absents par classe sur le train](figures/recap-titanic-ages.svg)

### À vous 1. Compter les absences et les survivants

Sur le train, comptez les valeurs manquantes de chaque colonne. Regroupez ensuite les passagers par classe de billet et calculez sur Survived trois colonnes nommées size, sum et mean. Le graphique suivant utilise votre tableau. Indices : `isna`, `groupby`, `agg`.

In [ ]:
# Repérer les mesures absentes sur les seules entrées du train.
absents = ...
assert isinstance(absents, pd.Series) and absents.index.equals(X.columns), (
    "Calculez un nombre par colonne de X."
)
assert absents.equals(X.isna().sum())
display(absents.to_frame("Valeurs absentes"))

In [ ]:
# Compter les passagers et les survivants de chaque classe de billet.
par_classe = ...
assert isinstance(par_classe, pd.DataFrame) and {"size", "sum", "mean"}.issubset(
    par_classe.columns
), "Utilisez trois agrégations sur Survived."
assert par_classe.index.tolist() == [1, 2, 3] and par_classe["size"].sum() == len(X)
assert par_classe["sum"].sum() == y.sum()
display(par_classe)

In [ ]:
# Observer la distribution des âges que le manifeste renseigne.
ax = X.Age.dropna().plot.hist(
    bins=np.arange(0, 91, 10), color="#317979", edgecolor="white"
)
ax.set(xlabel="Âge renseigné (ans)", ylabel="Passagers", title="Qui est représenté ?")
plt.show()

In [ ]:
# Comparer les taux de survie en changeant de dénominateur à chaque classe.
ax = (100 * par_classe["mean"]).plot.bar(color="#317979", rot=0)
ax.set(
    xlabel="Classe du billet",
    ylabel="Survivants (%)",
    ylim=(0, 100),
    title="Survie observée sur le train",
)
ax.bar_label(ax.containers[0], fmt="%.1f %%")
plt.show()

**À lire.** L’histogramme compte uniquement les âges renseignés. Dans le deuxième graphique, chaque barre divise le nombre de survivants de la classe par le nombre total de passagers de cette classe. Quelle information perdrait-on en supprimant tous les âges manquants ?

### Quels tarifs se recouvrent ?

![Distribution cumulée des tarifs pour les trois classes](figures/tp09-tarifs.svg)

Choisissez 50 % sur l’axe vertical, rejoignez chaque courbe, puis redescendez vers le tarif. Comparez vos lectures aux médianes calculées ci-dessous. Une valeur élevée mérite une vérification avant toute suppression.

In [ ]:
# Comparer la médiane et les extrêmes sans confondre une valeur rare et une erreur.
display(train.groupby("Pclass").Fare.agg(["min", "median", "max"]).round(2))

### À vous 2. Comparer des profils au sein de chaque classe

Le taux global en troisième classe peut cacher des écarts. Calculez les effectifs et le taux de survie par couple (Pclass, Sex). Sur la carte, repérez deux cases de la même colonne et commentez l’écart en donnant leurs effectifs. Tracez vous-même la carte avec `ax.imshow` sur le tableau des proportions, une échelle de 0 à 1 et la palette Blues.

In [ ]:
# Mesurer la survie au sein de chaque couple classe et sexe.
groupes = ...
assert isinstance(groupes, pd.DataFrame) and groupes.index.names == ["Pclass", "Sex"], (
    "Regroupez selon les deux colonnes, dans cet ordre."
)
assert {"size", "mean"}.issubset(groupes.columns) and groupes["size"].sum() == len(
    train
)
display(groupes)

In [ ]:
# Donner un dénominateur à chaque taux avant de lui attribuer une couleur.
taux = groupes["mean"].unstack("Pclass")
effectifs = groupes["size"].unstack("Pclass")
annotations = taux.map(lambda valeur: f"{valeur:.0%}") + "\nn=" + effectifs.astype(str)
display(effectifs.rename_axis(index="Sexe enregistré", columns="Classe du billet"))

Choisissez deux groupes dans une même classe. Quand vous lirez leur pourcentage sur la carte, combien de passagers comptera chaque groupe ?

In [ ]:
# Représenter la proportion et inscrire l’effectif dans chaque case.
fig, ax = plt.subplots(figsize=(6.2, 3), layout="constrained")
image = ...
assert image in ax.images, "Utilisez ax.imshow sur les proportions."
np.testing.assert_allclose(np.asarray(image.get_array()), taux.to_numpy())
for (row, col), rate in np.ndenumerate(taux.to_numpy(dtype=float)):
    ax.text(
        col,
        row,
        annotations.iat[row, col],
        ha="center",
        va="center",
        color="white" if float(rate) > 0.55 else "black",
    )
ax.set_xticks(range(len(taux.columns)), taux.columns)
ax.set_yticks(range(len(taux)), taux.index)
ax.set(
    xlabel="Classe du billet",
    ylabel="Sexe enregistré",
    title="Survie observée par groupe",
)
fig.colorbar(image, ax=ax, label="Proportion de survivants")
plt.show()

## 4. Data cleaning : apprendre les remplacements dans chaque pli

On contrôle le domaine des mesures et on conserve les âges manquants. Pour chaque colonne numérique, l’imputation remplacera les absences par la médiane **apprise sur les seules lignes d’entraînement du pli**. Les catégories manquantes ont une modalité explicite. On encode les catégories par des indicatrices (one-hot) et on standardise les mesures pour les modèles sensibles aux échelles. Pclass est traitée comme une catégorie.

![Effet de la médiane du train sur les âges du premier pli de validation](figures/tp09-imputation.svg)

Le segment orange correspond aux remplacements. D’où vient ce pic et combien de valeurs ont réellement été mesurées dans cette tranche ?

### À vous 3. Apprendre la médiane du bon côté

Ajustez un SimpleImputer sur Age dans la partie entraînement du premier pli. Transformez ensuite la validation avec cet objet. Indexez les lignes avec `iloc` et conservez une table à une colonne pour scikit-learn.

In [ ]:
# Séparer les âges connus pour apprendre de ceux à transformer.
from sklearn.impute import SimpleImputer

a, b = folds[0]
ages_train = X.iloc[a][["Age"]]
ages_val = X.iloc[b][["Age"]]
print(
    f"Âges disponibles pour apprendre la médiane : {ages_train.Age.count()} sur {len(ages_train)}."
)

In [ ]:
# Apprendre la valeur de remplacement du côté entraînement.
age_imputer = ...
assert isinstance(age_imputer, SimpleImputer) and hasattr(age_imputer, "statistics_"), (
    "Ajustez un SimpleImputer sur ages_train."
)
print(f"Médiane apprise : {age_imputer.statistics_[0]:g} ans.")

In [ ]:
# Comparer les mêmes passagers avant et après la transformation.
ages_complets = ...
assert (
    isinstance(ages_complets, np.ndarray) and ages_complets.shape == ages_val.shape
), "Transformez ages_val sans refaire de fit."
observes = ages_val.Age.notna().to_numpy()
np.testing.assert_allclose(ages_complets[observes, 0], ages_val.Age[observes])
np.testing.assert_allclose(ages_complets[~observes, 0], ages_train.Age.median())
avant_apres = ages_val.rename(columns={"Age": "Avant (ans)"}).assign(
    Après=ages_complets[:, 0]
)
display(
    pd.concat([avant_apres.loc[~observes].head(2), avant_apres.loc[observes].head(2)])
)
print(f"{(~observes).sum()} âges remplacés dans la validation.")

In [ ]:
# Décrire les opérations sans encore calculer de médiane sur les données.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

assert X.Pclass.isin([1, 2, 3]).all()
assert X[["Age", "Fare", "SibSp", "Parch"]].dropna(how="all").min().ge(0).all()
numeric = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
ports = make_pipeline(
    SimpleImputer(strategy="constant", fill_value="inconnu"),
    OneHotEncoder(handle_unknown="ignore", sparse_output=False),
)
categories = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
print("Traitement d’un port absent :", ports.steps[0][1].fill_value)

## 5. Feature engineering : représenter les proches à bord

Deux colonnes décrivent la famille. Pour un passager avec un conjoint et deux enfants, on obtient **3 proches à bord**. Le calcul additionne SibSp et Parch ; il ne compte pas le passager lui-même. L’arbre peut ainsi séparer les voyageurs seuls des voyageurs accompagnés en un seul test. On garde les deux colonnes originales pour distinguer les compositions familiales.

Le calcul est intégré au pipeline. Une nouvelle ligne aura besoin des mêmes sept champs bruts, et bénéficiera automatiquement de la même préparation.

### À vous 4. Construire la variable « proches »

Écrivez la transformation NumPy qui additionne les colonnes de chaque personne et conserve une colonne de sortie. Essayez-la sur les deux passagers ci-dessous. Réutilisez ensuite ce transformeur dans le pipeline : il doit aussi fonctionner après la sauvegarde.

In [ ]:
# Tester la transformation sur deux compositions familiales.
somme_proches = ...
assert isinstance(somme_proches, FunctionTransformer), (
    "Utilisez FunctionTransformer avec np.sum, axis et keepdims."
)
profils = np.array([[1, 2], [0, 0]])
resultat = somme_proches.transform(profils)
np.testing.assert_array_equal(resultat, [[3], [0]])
display(
    pd.DataFrame(profils, columns=["SibSp", "Parch"]).assign(Proches=resultat.ravel())
)

In [ ]:
# Distinguer petits groupes et familles nombreuses dans le manifeste.
nombre_proches = somme_proches.transform(X[["SibSp", "Parch"]].to_numpy()).ravel()
familles = train.groupby(np.minimum(nombre_proches, 4)).Survived.agg(["size", "mean"])
display(
    familles.rename(columns={"size": "Passagers", "mean": "Proportion de survivants"})
)

In [ ]:
# Lire un taux avec le nombre de passagers qui le composent.
ax = (100 * familles["mean"]).plot.bar(color="#317979", rot=0)
ax.set(
    xlabel="Nombre de proches (4 = quatre ou plus)",
    ylabel="Survivants (%)",
    ylim=(0, 100),
    title="Les groupes nombreux ont-ils le même profil ?",
)
ax.bar_label(ax.containers[0], labels=[f"n={n}" for n in familles["size"]])
plt.tight_layout()
plt.show()

In [ ]:
# Ajouter le nombre de proches par ligne à la préparation commune.
from sklearn.base import clone

proches = make_pipeline(
    FunctionTransformer(np.asarray),
    somme_proches,
    StandardScaler(),
)
prep = ColumnTransformer(
    [
        ("mesures", numeric, ["Age", "Fare", "SibSp", "Parch"]),
        ("port", ports, ["Embarked"]),
        ("classe_sexe", categories, ["Pclass", "Sex"]),
        ("proches", proches, ["SibSp", "Parch"]),
    ]
)
display(
    pd.Series(
        {nom: ", ".join(cols) for nom, _, cols in prep.transformers},
        name="Entrées par branche",
    )
)

## 6. Modeling : quelle valeur ajoutée par rapport à une règle triviale ?

Avec **LazyClassifier**, on essaie sur le premier pli une règle majoritaire, une régression logistique, un arbre, une forêt et un SVM. Notre critère est la **balanced accuracy**, la moyenne de deux proportions : la part des survivants reconnus et la part des non-survivants reconnus. Prédire la classe majoritaire pour tous donne 0,50 si les deux classes sont présentes. L’accuracy ordinaire peut être bien plus élevée.

Ce premier classement sert à explorer : un seul pli peut avantager un candidat. LazyClassifier ajoute sa propre préparation aux colonnes numériques qu’il reçoit. Pour une comparaison maîtrisée, la suite du TP refait l’évaluation des cinq méthodes avec notre pipeline et les quatre mêmes plis.

La logistique combine les entrées en un score linéaire. L’arbre apprend des conditions successives, la forêt combine plusieurs arbres. Le SVM à noyau permet une frontière courbe entre les deux classes. La règle majoritaire ignore les caractéristiques.

In [ ]:
# Sélectionner les cinq familles que LazyClassifier essaiera.
from lazypredict.Supervised import LazyClassifier
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

models = [
    DummyClassifier,
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier,
    SVC,
]
print(", ".join(model.__name__ for model in models))

In [ ]:
# Préparer le premier pli pour donner à chaque modèle les mêmes colonnes numériques.
a, b = folds[0]
first_prep = clone(prep).fit(X.iloc[a])
first_train = pd.DataFrame(first_prep.transform(X.iloc[a])).add_prefix("x")
first_val = pd.DataFrame(first_prep.transform(X.iloc[b])).add_prefix("x")
print(f"{first_train.shape[1]} colonnes après préparation. Aperçu de trois lignes :")
display(first_train.iloc[:3, :5])

In [ ]:
# Lire le premier classement avant de vérifier sa stabilité sur quatre plis.
screen = LazyClassifier(classifiers=models, random_state=seed, n_jobs=1)
ranking, _ = screen.fit(
    first_train, first_val, y.iloc[a].to_numpy(), y.iloc[b].to_numpy()
)
assert set(ranking.index) == {cls.__name__ for cls in models}
display(ranking[["Accuracy", "Balanced Accuracy"]])

### À vous 5. Comparer les modèles sur des billets inconnus

Complétez la fonction qui évalue un pipeline sur nos quatre plis. Elle doit retourner quatre balanced accuracies, une par validation. Utilisez `cross_val_score` avec le pipeline reçu, X, y et folds. La règle majoritaire sert de contrôle.

In [ ]:
# Calculer une mesure par pli, puis vérifier le cas de référence.
from sklearn.model_selection import cross_val_score  # noqa: F401


def evaluer_plis(estimator):
    return ...


reference = evaluer_plis(DummyClassifier(strategy="most_frequent"))
assert isinstance(reference, np.ndarray) and reference.shape == (4,), (
    "Renvoyez le tableau des quatre scores."
)
np.testing.assert_allclose(reference, 0.5)
display(
    pd.Series(
        reference, index=range(1, 5), name="Balanced accuracy majoritaire"
    ).rename_axis("Pli")
)

In [ ]:
# Comparer les mêmes familles sur quatre validations, avec imputation réapprise à chaque fois.


def pipeline(model):
    return Pipeline([("prep", clone(prep)), ("model", model)])


candidates = {
    "Majoritaire": DummyClassifier(strategy="most_frequent"),
    "Logistique": LogisticRegression(max_iter=1000),
    "Arbre": DecisionTreeClassifier(random_state=seed),
    "Forêt": RandomForestClassifier(random_state=seed, n_jobs=1),
    "SVM": SVC(),
}
print("Candidats :", ", ".join(candidates))

In [ ]:
# Comparer les modèles pli par pli, puis selon leur moyenne.
scores = pd.DataFrame(
    {label: evaluer_plis(pipeline(est)) for label, est in candidates.items()},
    index=[f"Pli {i}" for i in range(1, 5)],
).T
scores["moyenne"] = scores.mean(axis=1)
display(scores.sort_values("moyenne", ascending=False).round(3))

In [ ]:
# Afficher chaque validation pour comparer la dispersion et le classement moyen.
fig, ax = plt.subplots(figsize=(7.5, 3.4), layout="constrained")
for label, row in scores.iterrows():
    ax.scatter(row.iloc[:4], [label] * 4, color="#448985", alpha=0.7)
    ax.scatter([row["moyenne"]], [str(label)], marker="|", s=230, color="black")
ax.set(
    xlabel="Balanced accuracy (points = plis, trait = moyenne)",
    title="Un modèle arrive-t-il en tête dans chaque pli ?",
)
ax.grid(axis="x", alpha=0.2)
plt.show()

## 7. Grid Search ou Random Search à budget égal ?

Un **hyperparamètre** est un réglage choisi avant l’apprentissage. Pour isoler l’effet de la recherche, on prend la même famille (la forêt), les mêmes plis, le même critère et six configurations par méthode. La grille croise trois profondeurs et deux tailles minimales de feuille. La recherche aléatoire tire six couples distincts dans un espace plus fin. Une feuille plus grande limite les décisions fondées sur quelques passagers.

Chaque méthode ajuste 6 × 4 = 24 forêts pour la validation. L’étendue des espaces diffère volontairement. Le gagnant décrit cette expérience et cette graine. Pour le choix final, les autres modèles du comparatif restent en lice.

### À vous 6. Donner le même budget aux deux recherches

Définissez une grille avec des profondeurs 3, 6 ou illimitée et une feuille de taille minimale 1 ou 5. Calculez son nombre de configurations avec ParameterGrid. La recherche aléatoire utilisera ce nombre comme budget.

In [ ]:
# Construire le plan de recherche de la forêt et en compter les combinaisons.
from sklearn.model_selection import ParameterGrid

grille_foret = ...
assert isinstance(grille_foret, dict) and set(grille_foret) == {
    "model__max_depth",
    "model__min_samples_leaf",
}, "Préfixez les paramètres de l’étape model par model__."
assert set(grille_foret["model__max_depth"]) == {3, 6, None} and set(
    grille_foret["model__min_samples_leaf"]
) == {1, 5}
combinaisons = pd.DataFrame(ParameterGrid(grille_foret))
combinaisons.columns = ["Profondeur", "Feuille"]
combinaisons["Profondeur"] = (
    combinaisons["Profondeur"].astype("Int64").astype("string").fillna("illimitée")
)
display(combinaisons)

In [ ]:
# Compter les configurations, puis les ajustements sur quatre plis.
budget = ...
assert budget == 6, "Comptez les combinaisons, puis affectez ce nombre à budget."
print(budget * len(folds), "ajustements de validation par méthode")

In [ ]:
# Conserver les mêmes conditions pour les deux recherches.
from time import perf_counter

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

settings = {
    "estimator": pipeline(candidates["Forêt"]),
    "cv": folds,
    "scoring": "balanced_accuracy",
    "n_jobs": 1,
    "refit": False,
    "error_score": "raise",
}
print(f"Critère : {settings['scoring']}. Plis par configuration : {len(folds)}.")

In [ ]:
# Garder pour chaque configuration ses paramètres et sa moyenne sur les quatre plis.
def essais(recherche):
    resultats = pd.DataFrame(recherche.cv_results_["params"])
    resultats.columns = [
        "Profondeur" if c.endswith("max_depth") else "Feuille" for c in resultats
    ]
    resultats["Profondeur"] = (
        resultats["Profondeur"].astype("Int64").astype("string").fillna("∞")
    )
    resultats["Balanced accuracy"] = recherche.cv_results_["mean_test_score"]
    return resultats


def mesurer(recherche):
    debut = perf_counter()
    recherche.fit(X, y)
    return perf_counter() - debut

In [ ]:
# Essayer toutes les combinaisons de la grille et lire le réglage en tête.
grille = GridSearchCV(param_grid=grille_foret, **settings)
temps_grille = mesurer(grille)
display(essais(grille).sort_values("Balanced accuracy", ascending=False).round(3))

La grille a essayé deux tailles de feuille pour chaque profondeur. Avec le même budget, quelles profondeurs et quelles tailles de feuille le tirage aléatoire explore-t-il ?

In [ ]:
# Tirer six couples dans un espace plus fin et afficher les couples réellement essayés.
profondeurs = [2, 3, 4, 5, 6, 8, 10, None]
feuilles = [1, 2, 3, 5, 8, 12]
aleatoire = RandomizedSearchCV(
    param_distributions={
        "model__max_depth": profondeurs,
        "model__min_samples_leaf": feuilles,
    },
    n_iter=budget,
    random_state=seed,
    **settings,
)
temps_aleatoire = mesurer(aleatoire)
display(essais(aleatoire).sort_values("Balanced accuracy", ascending=False).round(3))

In [ ]:
# Comparer les deux meilleurs réglages à budget égal et les garder pour la sélection finale.
searches = {"Grille": grille, "Aléatoire": aleatoire}
comparaison = []
for nom, recherche in searches.items():
    parametres = {
        k.removeprefix("model__"): v for k, v in recherche.best_params_.items()
    }
    candidates[nom] = clone(candidates["Forêt"]).set_params(**parametres)
    comparaison.append(essais(recherche).iloc[recherche.best_index_])
bilan = pd.DataFrame(comparaison, index=list(searches))
bilan["Essais"] = [len(essais(recherche)) for recherche in searches.values()]
bilan["Secondes sur ce poste"] = [temps_grille, temps_aleatoire]
display(bilan.round(3))

Les deux méthodes ont essayé six couples. Sur les cartes, une case blanche indique un couple jamais essayé. Repérez une profondeur explorée par une seule méthode. La couleur suit la même échelle sur les deux cartes.

In [ ]:
# Disposer les scores sur les mêmes axes pour les deux recherches.
def carte_essais(recherche):
    table = essais(recherche).pivot(
        index="Feuille", columns="Profondeur", values="Balanced accuracy"
    )
    return table.reindex(
        index=feuilles, columns=["∞" if p is None else str(p) for p in profondeurs]
    )


bornes = pd.concat([essais(grille), essais(aleatoire)])["Balanced accuracy"].agg(
    ["min", "max"]
)
print(f"Échelle de couleur commune : de {bornes['min']:.3f} à {bornes['max']:.3f}.")

In [ ]:
# Comparer les zones explorées dans les deux espaces de recherche.
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), sharey=True, layout="constrained")
for ax, (nom, recherche) in zip(axes, searches.items(), strict=True):
    table = carte_essais(recherche)
    image = ax.imshow(table, vmin=bornes["min"], vmax=bornes["max"], aspect="auto")
    ax.set_xticks(range(len(table.columns)), table.columns)
    ax.set_yticks(range(len(table)), table.index)
    ax.set(xlabel="Profondeur maximale (∞ = illimitée)", title=nom)
axes[0].set_ylabel("Taille minimale d’une feuille")
fig.colorbar(image, ax=axes, label="Balanced accuracy moyenne")
plt.show()

## 8. Évaluer une seule fois sur les billets réservés

On choisit la meilleure moyenne de validation parmi les modèles initiaux et les deux forêts réglées. En cas d’égalité, on conserve le premier candidat dans l’ordre du tableau de définition. On le réentraîne sur tout le train puis on ouvre le test. Les prédictions suivent la règle par défaut du classifieur. Modifier le modèle après avoir vu ce bilan demanderait un nouveau test indépendant.

### À vous 7. Figer et défendre le choix avant le test

Rassemblez les moyennes des cinq modèles et les meilleurs scores CV des deux recherches dans selection (fourni). Écrivez la sélection du nom associé à la plus grande valeur et justifiez le choix en une phrase : citez la métrique et un concurrent. À égalité exacte, gardez le premier candidat.

In [ ]:
# Classer ensemble les cinq modèles initiaux et les deux forêts réglées.
selection = scores["moyenne"].to_dict()
selection.update({label: search.best_score_ for label, search in searches.items()})
display(pd.Series(selection, name="balanced accuracy CV").sort_values(ascending=False))

In [ ]:
# Figer le nom du modèle et justifier la décision avant de regarder le test.
winner = ...
justification = ...
assert winner in selection and np.isclose(selection[winner], max(selection.values())), (
    "Choisissez un nom dont le score est maximal."
)
assert isinstance(justification, str) and len(justification.split()) >= 6, (
    "Notez votre justification avant de passer au test."
)
print(winner, justification)

In [ ]:
# Réajuster le candidat retenu sur tous les billets de développement.
fitted = pipeline(candidates[winner]).fit(X, y)
print(
    f"{winner} ajusté sur {len(X)} passagers. Balanced accuracy en validation : {selection[winner]:.3f}."
)

In [ ]:
# Prédire sur les champs bruts du test avant de consulter les verdicts.
pred = fitted.predict(test[inputs])
display(test[["PassengerId"]].assign(Prédiction=pred).head())

In [ ]:
# Mesurer les deux accuracies sur les mêmes passagers réservés.
from sklearn.metrics import accuracy_score, balanced_accuracy_score

print(f"Passagers bien classés : {(test.Survived == pred).sum()} sur {len(pred)}.")
print(f"Accuracy : {accuracy_score(test.Survived, pred):.3f}")
print(f"Balanced accuracy : {balanced_accuracy_score(test.Survived, pred):.3f}")

Quelle classe est la plus difficile à reconnaître ? Comparez les deux rappels. Le support est le nombre de passagers réellement présents dans chaque classe.

In [ ]:
# Distinguer le bilan de chaque classe du score global.
from sklearn.metrics import classification_report

print(
    classification_report(
        test.Survived,
        pred,
        target_names=["Non-survivant", "Survivant"],
        digits=3,
        zero_division=0,
    )
)

In [ ]:
# Retrouver les erreurs individuelles derrière les taux.
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    test.Survived,
    pred,
    display_labels=["Non-survivant", "Survivant"],
    cmap="Blues",
    colorbar=False,
)
plt.xlabel("Classe prédite")
plt.ylabel("Classe observée")
plt.title("Chaque passager du test apparaît dans une seule case")
plt.show()

### À vous 8. Retrouver le rappel sur la matrice

Repérez la ligne des survivants dans la matrice. Comptez les survivants reconnus et manqués à partir de pred et des étiquettes du test, puis calculez le rappel. Expliquez comment une accuracy supérieure à 80 % peut coexister avec ce résultat.

In [ ]:
# Reconstituer la métrique à partir des passagers plutôt que recopier le rapport.
survivants = test.Survived.to_numpy() == 1
reconnus = ...
manques = ...
assert isinstance(reconnus, (int, np.integer)) and isinstance(
    manques, (int, np.integer)
), "Comptez les passagers de la deuxième ligne."
assert reconnus + manques == survivants.sum() and reconnus == pred[survivants].sum()
print(f"Parmi les survivants : {reconnus} reconnus et {manques} manqués.")

In [ ]:
# Choisir le dénominateur correspondant au rappel des survivants.
rappel = ...
assert isinstance(rappel, (float, np.floating)) and np.isclose(
    rappel, reconnus / survivants.sum()
), "Divisez par le nombre total de survivants observés."
print(f"{reconnus} reconnus, {manques} manqués, rappel {rappel:.1%}.")

**Lecture.** Sur la ligne « Survivant », la case de gauche compte les survivants classés à tort parmi les non-survivants. Le *rappel* divise les survivants reconnus par tous les survivants observés ; la *précision* les divise par toutes les personnes prédites survivantes. Le faible effectif et le voyage unique limitent la généralisation. Comme chez Nervure, le score de sélection et le bilan réservé répondent à deux questions différentes.

## 9. Sauvegarder, recharger et prédire sur un manifeste brut

On conserve dans un même fichier les transformations apprises, les colonnes attendues et le modèle. Les versions des bibliothèques et la graine facilitent la reproduction. Le nom dépend du contenu, ce qui permet de distinguer les versions. Rechargez uniquement vos propres fichiers joblib avec l’environnement verrouillé du dépôt.

In [ ]:
# Préciser le contrat du fichier qui servira pour une nouvelle prédiction.
from importlib.metadata import version

bundle = {
    "pipeline": fitted,
    "inputs": inputs,
    "modèle": winner,
    "seed": seed,
    "cible": "Survived (1 = survie)",
    "versions": {p: version(p) for p in ["scikit-learn", "numpy", "pandas", "joblib"]},
}
print("Cible :", bundle["cible"])
print("Champs à fournir :", ", ".join(bundle["inputs"]))

In [ ]:
# Écrire le fichier et en donner le nom pour pouvoir le retrouver.
import hashlib
from io import BytesIO

import joblib

folder = root / "artefacts" / "recap_titanic"
folder.mkdir(exist_ok=True, parents=True)
buffer = BytesIO()
joblib.dump(bundle, buffer)
payload = buffer.getvalue()
path = folder / f"pipeline-{hashlib.sha256(payload).hexdigest()[:16]}.joblib"
path.write_bytes(payload)
print(f"{path.name} ({path.stat().st_size:,} octets)")

### À vous 9. Vérifier le modèle réellement livré

Rechargez le fichier joblib à partir de path. Faites prédire le pipeline rechargé sur les champs bruts du test dans l’ordre enregistré. Comparez ses décisions à pred. Quel problème ce contrôle permet-il de détecter, et quelle information le test de performance apporte-t-il en plus ?

In [ ]:
# Lire les métadonnées du fichier écrit sur disque.
restored = ...
assert isinstance(restored, dict) and "pipeline" in restored, (
    "Chargez le fichier écrit par la cellule précédente."
)
display(pd.Series(restored["versions"], name="Version enregistrée"))

In [ ]:
# Rejouer les décisions et compter les différences avec le modèle en mémoire.
rejouees = ...
assert isinstance(rejouees, np.ndarray) and rejouees.shape == pred.shape, (
    "Prédisez sur les mêmes lignes brutes avec le pipeline rechargé."
)
np.testing.assert_array_equal(rejouees, pred)
print(f"Rejeu vérifié sur {len(pred)} passagers, sans nouvel entraînement.")

In [ ]:
# Traiter un passager fictif dont l’âge manque en utilisant le pipeline rechargé.
example = pd.DataFrame([[3, "male", np.nan, 1, 0, 12.5, "S"]], columns=inputs)
display(example.assign(Prédiction=restored["pipeline"].predict(example)))

## Défis facultatifs

1. **Quantifier la baseline.** Sur les quatre plis, quelle accuracy ordinaire obtient la règle majoritaire ? Expliquez pourquoi sa balanced accuracy reste à 0,50. Partez de `pipeline(candidates["Majoritaire"])` et de `cross_val_score(..., cv=folds, scoring="accuracy")`.
2. **Tester l’apport des proches.** Avec la même régression logistique et les mêmes plis, retirez uniquement la branche `proches` du prétraitement. Partez de `pipeline(candidates["Logistique"]).set_params(prep__proches="drop")` et comparez les écarts pli par pli. Pourquoi un indicateur construit comme une somme peut-il apporter peu à un modèle linéaire ?

[Solutions expliquées et résultats](corrections/TP09_pipeline_ML_Titanic.ipynb)

In [ ]:
# Facultatif 1 : calculez les quatre accuracies de la règle majoritaire.
# Comparez-les ensuite à la ligne Majoritaire du tableau scores.

In [ ]:
# Facultatif 2 : retirez la branche proches d'une copie de la logistique.
# Calculez quatre scores, puis soustrayez-les aux scores de la version complète.

**Pour approfondir** : [Pipelines scikit-learn](https://scikit-learn.org/stable/modules/compose.html), [recherche d’hyperparamètres](https://scikit-learn.org/stable/modules/grid_search.html) et [Lazy Predict](https://github.com/shankarpandala/lazypredict).